# 벚나무 핵심 전처리

평비·발아일·만발일은 제외하고, 프로젝트에 필요한 **지점·년도·개화일·개화일_DOY**만 만듭니다.

## 1. 원본 파일 업로드
아래 셀을 실행한 뒤 **`OBS_계절관측_벚나무.csv`** 파일을 업로드하세요.

In [ ]:
from google.colab import files

uploaded = files.upload()
expected_file = "OBS_계절관측_벚나무.csv"

if expected_file not in uploaded:
    raise FileNotFoundError(
        f"업로드한 파일 이름이 '{expected_file}'와 다릅니다. "
        f"현재 업로드 파일: {list(uploaded.keys())}"
    )

print(f"업로드 완료: {expected_file}")

## 2. 전처리 실행

In [ ]:
from pathlib import Path
import pandas as pd

# 설정
INPUT_FILE = Path("OBS_계절관측_벚나무.csv")
OUTPUT_FILE = Path("벚나무_분석용.csv")
REVIEW_FILE = Path("벚나무_제외목록.csv")
START_YEAR, END_YEAR = 2016, 2025

def read_csv_with_encoding(path, **kwargs):
    """CP949, UTF-8 순서로 파일 인코딩을 시도한다."""
    for encoding in ("cp949", "utf-8-sig", "utf-8"):
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"파일 인코딩을 읽지 못했습니다: {path}")

def clean_text(series):
    """공백과 결측 표기를 통일한다."""
    result = series.astype("string").str.strip()
    missing_tokens = {"", "-", "―", "관측 안됨", "관측안됨", "nan", "NaN", "None"}
    return result.mask(result.isin(missing_tokens), pd.NA)

# 원본은 2행 헤더 구조이므로 필요한 열만 이름을 지정해 읽는다.
raw = read_csv_with_encoding(
    INPUT_FILE,
    skiprows=2,
    header=None,
    names=["지점", "년도", "발아일", "발아평비", "개화일", "개화평비", "만발일", "만발평비"],
    dtype="string",
).dropna(how="all")

# 프로젝트 분석에 필요한 열만 남긴다.
result = pd.DataFrame({
    "지점": clean_text(raw["지점"]),
    "년도": pd.to_numeric(clean_text(raw["년도"]), errors="coerce").astype("Int64"),
    "벚나무_개화일": pd.to_datetime(clean_text(raw["개화일"]), errors="coerce"),
})

# 2016~2025년만 사용한다.
result = result.loc[
    result["년도"].between(START_YEAR, END_YEAR, inclusive="both")
].copy()

# 날짜가 있는 행에만 연중일수(DOY)를 만든다.
result["벚나무_개화일_DOY"] = result["벚나무_개화일"].dt.dayofyear.astype("Int64")

# 분석에 쓸 수 없는 행은 별도 목록으로 보관한다.
invalid = (
    result["지점"].isna()
    | result["년도"].isna()
    | result["벚나무_개화일"].isna()
)
review = result.loc[invalid].copy()
result = result.loc[~invalid].copy()

# 같은 지점·연도가 중복되면 병합하면 안 되므로 오류를 낸다.
duplicated = result.duplicated(["지점", "년도"], keep=False)
if duplicated.any():
    raise ValueError(
        "지점+년도 중복이 있습니다. 원본을 확인하세요.\n"
        + result.loc[duplicated, ["지점", "년도"]].to_string(index=False)
    )

# CSV 저장용 날짜 형식
result["벚나무_개화일"] = result["벚나무_개화일"].dt.strftime("%Y-%m-%d")

result = result.sort_values(["지점", "년도"]).reset_index(drop=True)
review = review.sort_values(["지점", "년도"]).reset_index(drop=True)

result.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
review.to_csv(REVIEW_FILE, index=False, encoding="utf-8-sig")

print(f"분석용 데이터: {OUTPUT_FILE} / {len(result):,}행")
print(f"제외 목록: {REVIEW_FILE} / {len(review):,}행")
result.head()

## 3. 결과 파일 다운로드

In [ ]:
import zipfile
from google.colab import files

output_files = ['벚나무_분석용.csv', '벚나무_제외목록.csv']
zip_name = "전처리결과.zip"

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_name in output_files:
        zf.write(file_name)

print("생성된 파일:", output_files)
files.download(zip_name)